<center>
<img src="../images/fscampus_small2.png" width="1200"/>
</center>

<center>

# Investments

***Finance 2 - BFIN***

**Dr. Omer Cayirli**

Lecturer in Empirical Finance

omer_cayirli@uncbusiness.net
</center>

---

## Lecture 07

---


### Outline

*   Theoretical Limitations of CAPM
    *   Human Capital and Liquidity Risk
*   The General Multifactor Framework
*   Arbitrage Pricing Theory (APT)
*   Empirical Models
    *   The Fama-French and Carhart Models
*   Smart Beta and Performance Evaluation

---



### Why Multifactor Models?

* Under CAPM, expected excess returns are proportional to market beta:
    $$E(r_i)-r_f=\beta_{i,M}[E(r_M)-r_f].$$

* Historical studies have found size and value patterns in average returns that market beta alone does not fully explain.

* The single-index model attributes return co-movement to the market alone. Other common influences, such as interest rates and inflation, motivate multifactor models.

CAPM prices expected returns; the single-index model describes return variation. Additional return co-movement does not, by itself, disprove CAPM.

---


### Theoretical Extensions of CAPM

* Human capital
    * Labor income is part of investors' wealth but is largely nontradable. Investors may prefer financial assets that hedge their labor-income risk.

* Changing investment opportunities (ICAPM)
    * Investors value assets that hedge adverse changes in future investment opportunities and may accept lower expected returns for that protection (Merton, 1973).

* Liquidity risk
    * Assets that perform poorly when market-wide liquidity falls require compensation for that exposure (Pástor and Stambaugh, 2003).

---


### Generalizing to Multifactor Models

Excess returns, $R_i = r_i - r_f$, reflect $K$ common factor surprises and a firm-specific shock:

$$
R_i = E(R_i) + \beta_{i1}F_1 + \beta_{i2}F_2 + \dots + \beta_{iK}F_K + e_i
$$

*   $F_k$: Factor $k$'s surprise relative to expectations, with $E(F_k) = 0$.
*   $\beta_{ik}$: Asset $i$'s return sensitivity to a one-unit surprise in factor $k$, holding other factors fixed.
*   $e_i$: Firm-specific shock, with $E(e_i) = 0$, uncorrelated with the factors.

For simplicity, assume residuals are also uncorrelated across assets.

---



### A Two-Factor Economy Example

Returns respond to surprises in GDP growth ($F_{GDP}$) and interest rates ($F_{IR}$). A positive $F_{IR}$ denotes an unexpected rate increase.

$$
R_i = E(R_i) + \beta_{i,GDP}F_{GDP} + \beta_{i,IR}F_{IR} + e_i
$$

Assume these illustrative sensitivities, with returns and factor surprises measured in percentage points:

| Firm | Activity | $\beta_{GDP}$ | $\beta_{IR}$ |
| :--- | :--- | ---: | ---: |
| Alpha | Cyclical manufacturer | 1.5 | -0.2 |
| Beta | Regulated utility | 0.2 | -1.5 |

An unexpected rate increase hurts Beta more, holding other shocks fixed. Similar market betas can conceal these differences in factor exposure.

---



In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


def calculate_factor_scenario(alpha_gdp, alpha_rates, beta_gdp, beta_rates,
                              gdp_surprise, rate_surprise):
    """Return surprises in pp; expected returns are excluded and residuals are zero."""
    # Factor SDs are 1 pp, with zero correlation. The diversified market has
    # no residual shock; firm residuals are uncorrelated with the factors.
    factor_cov = np.eye(2)
    market_loadings = np.array([1.0, -0.5])
    loadings = np.array([[alpha_gdp, alpha_rates], [beta_gdp, beta_rates]])
    shocks = np.array([gdp_surprise, rate_surprise])

    # Population market beta = Cov(R_i, R_M) / Var(R_M).
    market_variance = market_loadings @ factor_cov @ market_loadings
    market_betas = (loadings @ factor_cov @ market_loadings) / market_variance
    market_surprise = float(market_loadings @ shocks)
    contributions = loadings * shocks
    return {
        'contributions': contributions,
        'multifactor': contributions.sum(axis=1),
        'single_index': market_betas * market_surprise,
        'market_betas': market_betas,
        'market_surprise': market_surprise,
    }


def plot_factor_scenario(data, gdp_surprise, rate_surprise):
    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(13, 5.5), sharey=True, constrained_layout=True)
    x, width = np.arange(2), 0.30
    names = ['Alpha', 'Beta']

    # Separate bars keep positive and negative contributions visible from zero.
    ax1.bar(x - width / 2, data['contributions'][:, 0], width,
            label='GDP', color='#0072B2')
    ax1.bar(x + width / 2, data['contributions'][:, 1], width,
            label='Rates', color='#D55E00')
    ax1.scatter(x, data['multifactor'], color='#111111', s=55,
                zorder=5, label='Net effect')
    for xpos, value in zip(x, data['multifactor']):
        ax1.annotate(f'{value:+.2f}', (xpos, value),
                     xytext=(0, 9 if value >= 0 else -16),
                     textcoords='offset points', ha='center', fontsize=10)
    ax1.set_title('Factor contributions\n'
                  f'GDP surprise: {gdp_surprise:+.2f} pp | '
                  f'Rates: {rate_surprise:+.2f} pp', fontsize=11, fontweight='normal')
    ax1.set_xticks(x, names)
    ax1.set_ylabel('Return surprise (percentage points)')

    multifactor_bars = ax2.bar(x - width / 2, data['multifactor'], width,
                               label='Multifactor', color='#333333')
    single_index_bars = ax2.bar(x + width / 2, data['single_index'], width,
                                label='Single-index', color='#BDBDBD',
                                edgecolor='#555555', hatch='//')
    for bars in (multifactor_bars, single_index_bars):
        ax2.bar_label(bars, fmt='%+.2f', padding=4, fontsize=10)
    ax2.set_title('Multifactor and single-index estimates\n'
                  f"Market surprise: {data['market_surprise']:+.2f} pp",
                  fontsize=11, fontweight='normal')
    ax2.set_xticks(x, [f'{name}\n(Market beta: {beta:.2f})'
                       for name, beta in zip(names, data['market_betas'])])
    for ax in (ax1, ax2):
        ax.axhline(0, color='#333333', linewidth=0.8)
        ax.set_axisbelow(True)
        ax.grid(axis='y', linestyle=':', alpha=0.4)
        # Fixed across redraws; bounded shocks keep all outcomes within +/-12 pp.
        # The 3 pp margin leaves room for labels at the extremes.
        ax.set_ylim(-15, 15)
        ax.set_yticks(np.arange(-15, 16, 5))
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15),
                  ncol=3 if ax is ax1 else 2, frameon=False)
    return fig


def make_factor_scenario_dashboard():
    # Local controls and callback remain separate from the lecture's other widgets.
    def slider(value, lower, upper, step, description):
        return widgets.FloatSlider(
            value=value, min=lower, max=upper, step=step,
            description=description, readout_format='.2f', continuous_update=False,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='98%'))

    alpha_gdp = slider(1.5, -2, 3, 0.1, 'GDP beta:')
    alpha_rates = slider(-0.2, -2, 2, 0.1, 'Rates beta:')
    beta_gdp = slider(0.2, -2, 3, 0.1, 'GDP beta:')
    beta_rates = slider(-1.5, -3, 1, 0.1, 'Rates beta:')
    gdp_surprise = slider(0.5, -2, 2, 0.5, 'GDP surprise (pp):')
    rate_surprise = slider(2.0, -2, 2, 0.25, 'Rate surprise (pp):')
    controls = [alpha_gdp, alpha_rates, beta_gdp, beta_rates,
                gdp_surprise, rate_surprise]
    plot_output = widgets.Output()

    def update(change=None):
        data = calculate_factor_scenario(*(control.value for control in controls))
        with plot_output:
            clear_output(wait=True)
            fig = plot_factor_scenario(data, gdp_surprise.value, rate_surprise.value)
            plt.show()
            plt.close(fig)

    for control in controls:
        control.observe(update, names='value')

    def column(children):
        return widgets.VBox(children, layout=widgets.Layout(width='50%'))

    dashboard = widgets.VBox([
        widgets.HTML('<h3 style="font-weight: normal;">Scenario analysis: factor sensitivities</h3>'),
        widgets.HTML(
            '<p>Factor standard deviations: 1 pp each; correlation: 0.<br>'
            'Diversified market surprise = GDP surprise &minus; 0.5 &times; rate surprise.<br>'
            'Charts show deviations from expected returns, with firm-specific shocks set to zero.</p>'),
        widgets.HBox([
            column([widgets.HTML('Alpha (cyclical manufacturer)'), alpha_gdp, alpha_rates]),
            column([widgets.HTML('Beta (regulated utility)'), beta_gdp, beta_rates]),
        ]),
        widgets.HBox([column([gdp_surprise]), column([rate_surprise])]),
        plot_output,
    ])
    update()
    return dashboard


factor_scenario_dashboard = make_factor_scenario_dashboard()
display(factor_scenario_dashboard)


### The Law of One Price and Arbitrage

*   Arbitrage
    *   An arbitrage opportunity arises when an investor can earn a riskless profit without making a net investment.
    *   It involves the simultaneous purchase and sale of equivalent securities to profit from price discrepancies.

*   The Law of One Price
    *   If two assets are equivalent in all economically relevant respects, they should have the same market price.
    *   Violation of this law implies an arbitrage opportunity exists.

*   Market Equilibrium via Arbitrage
    *   If an arbitrage opportunity exists, investors will buy the underpriced asset and sell the overpriced asset.
    *   Because the strategy is riskless and requires zero capital, investors will scale these positions largely.
    *   This buying and selling pressure forces prices back into alignment, quickly eliminating the profit opportunity.
    *   The Arbitrage Pricing Theory (APT) relies on the condition that well-functioning markets rule out arbitrage opportunities.

---

### Diversification and Idiosyncratic Risk

*   The Arbitrage Pricing Theory relies on the operation of the Law of Large Numbers in portfolio construction.
*   Well-Diversified Portfolios
    *   As the number of assets in a portfolio increases ($N \to \infty$), the non-systematic (idiosyncratic) components of return cancel each other out.
    *   For a sufficiently well-diversified portfolio $P$, the residual variance approaches zero ($\sigma^2(e_P) \to 0$).
*   Systematic Risk Dominance
    *   Because the idiosyncratic error term vanishes, the realized return on a well-diversified portfolio is determined almost exclusively by its expected return and its systematic factor sensitivities.
    $$
    R_P \approx E(R_P) + \beta_P F
    $$
    *   Consequently, only systematic risk remains relevant for pricing. The following no-arbitrage argument uses well-diversified portfolios.

---

### APT: The Single Factor Logic

Consider two well-diversified portfolios, Alpha and Beta, whose returns are driven by a single common factor, $F$. For this idealized arbitrage argument, assume their idiosyncratic risk is zero.

$$
R_{\mathrm{Alpha}} = E(R_{\mathrm{Alpha}}) + \beta_{\mathrm{Alpha}} F
$$
$$
R_{\mathrm{Beta}} = E(R_{\mathrm{Beta}}) + \beta_{\mathrm{Beta}} F
$$

Suppose both portfolios have the same sensitivity to the factor, such that $\beta_{\mathrm{Alpha}} = \beta_{\mathrm{Beta}} = 1$.

*   The Arbitrage Argument
    *   Assume for a moment that their expected excess returns differ: $E(R_{\mathrm{Alpha}}) = 10\%$ and $E(R_{\mathrm{Beta}}) = 8\%$.
    *   An investor can construct a zero-net-investment strategy by buying \$100 of Alpha and short-selling \$100 of Beta.
    *   Net Risk Exposure: $100 \cdot F - 100 \cdot F = 0$. The factor risk cancels out perfectly.
    *   Net Profit: $\$100 \times (0.10 - 0.08) = \$2$.

*   Conclusion
    *   This strategy yields a riskless profit of \$2 with zero net investment.
    *   Investors would exploit this opportunity, driving up the price of Alpha (lowering its return) and driving down the price of Beta (raising its return) until $E(R_{\mathrm{Alpha}}) = E(R_{\mathrm{Beta}})$.
    *   Therefore, in the absence of arbitrage, these equal-beta portfolios must have equal expected returns.

---

### The No-Arbitrage Equation

*   Risk Premium and Beta
    *   The arbitrage argument extends to portfolios with different sensitivities to the common factor.
    *   For well-diversified portfolios with nonzero betas, no arbitrage requires a common ratio of expected excess return to beta.

$$
\frac{E[R_{\mathrm{Alpha}}]}{\beta_{\mathrm{Alpha}}} = \frac{E[R_{\mathrm{Beta}}]}{\beta_{\mathrm{Beta}}} = \lambda
$$

*   The Factor Risk Premium ($\lambda$)
    *   The constant $\lambda$ is called the Factor Risk Premium.
    *   It represents the additional expected excess return an investor demands for bearing one unit of that specific systematic risk.
    *   For a portfolio with a beta of 1, the expected excess return is exactly $\lambda$.

---

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


def calculate_apt_scenario(beta, residual_sd, gap):
    """Total returns in percent; factor surprises, residual SD, and gap in pp."""
    factor = np.linspace(-3.0, 3.0, 100)
    risk_free, factor_premium = 4.0, 6.0
    expected_alpha = risk_free + beta * factor_premium
    expected_beta = expected_alpha + gap
    alpha_returns = expected_alpha + beta * factor
    beta_returns = expected_beta + beta * factor

    # The same standardized draws are reused; changing a slider adds no new noise.
    residual_draws = np.random.default_rng(42).standard_normal(factor.size)
    single_returns = alpha_returns + residual_sd * residual_draws
    if gap > 0:
        trade = 'Buy Beta; short Alpha'
    elif gap < 0:
        trade = 'Buy Alpha; short Beta'
    else:
        trade = 'No arbitrage gap'
    return {
        'beta': beta, 'residual_sd': residual_sd, 'gap': gap,
        'risk_free': risk_free, 'factor_premium': factor_premium,
        'factor': factor, 'single_returns': single_returns,
        'alpha_returns': alpha_returns, 'beta_returns': beta_returns,
        'expected_alpha': expected_alpha, 'expected_beta': expected_beta,
        'trade': trade, 'profit_per_100': abs(gap),
    }


def plot_apt_scenario(data):
    fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharey=True,
                             constrained_layout=True)
    ax1, ax2, ax3 = axes
    factor, beta, gap = data['factor'], data['beta'], data['gap']
    alpha_color, beta_color = '#0072B2', '#D55E00'
    fig.suptitle(
        f"Risk-free rate: {data['risk_free']:.0f}% | "
        f"Factor premium: {data['factor_premium']:.0f}% | Common beta: {beta:.1f}",
        fontsize=12, fontweight='normal')

    ax1.scatter(factor, data['single_returns'], color='#9BD0EA',
                edgecolors=alpha_color, s=25, alpha=0.8, label='Single asset')
    ax1.plot(factor, data['alpha_returns'], color=alpha_color, linewidth=2.5,
             label='Alpha (diversified)')
    ax1.set_title('1. Diversification\n'
                  f"Residual SD: {data['residual_sd']:.1f} pp",
                  fontsize=11, fontweight='normal')

    ax2.plot(factor, data['alpha_returns'], color=alpha_color,
             linewidth=2.5, label='Alpha')
    ax2.plot(factor, data['beta_returns'], color=beta_color,
             linewidth=2.5, linestyle='--', label='Beta')
    if gap != 0:
        ax2.fill_between(factor, data['alpha_returns'], data['beta_returns'],
                         color='#E69F00', alpha=0.25, label='Return spread')
        trade_note = (data['trade'] + '\n'
                      + rf"Profit: \${data['profit_per_100']:.2f} per \$100 on each leg")
    else:
        trade_note = 'No arbitrage gap\nEqual expected returns; zero profit'
    ax2.text(0.04, 0.05, trade_note, transform=ax2.transAxes, fontsize=10,
              va='bottom', bbox=dict(facecolor='white', edgecolor='none', alpha=0.9))
    ax2.set_title('2. Equal-beta portfolios\n'
                  f'Gap (Beta - Alpha): {gap:+.1f} pp', fontsize=11, fontweight='normal')

    betas = np.linspace(0, 2.2, 100)
    ax3.plot(betas, data['risk_free'] + betas * data['factor_premium'],
             color='#333333', linewidth=2, label='APT pricing line')
    ax3.scatter(beta, data['expected_alpha'], color=alpha_color, s=65,
                zorder=5, label='Alpha')
    ax3.scatter(beta, data['expected_beta'], color=beta_color, marker='x',
                linewidths=2, s=85, zorder=6, label='Beta')
    if gap != 0:
        ax3.annotate('', xy=(beta, data['expected_alpha']),
                     xytext=(beta, data['expected_beta']),
                     arrowprops=dict(arrowstyle='->', color='#555555',
                                     linewidth=1.5, shrinkA=0, shrinkB=0,
                                     mutation_scale=min(9, max(3, abs(gap) * 3))))
        adjustment_note = ('Expected-return adjustment\n'
                           f"Beta: {data['expected_beta']:.1f}% to "
                           f"{data['expected_alpha']:.1f}%")
    else:
        adjustment_note = 'Both portfolios lie\non the APT pricing line'
    ax3.text(0.04, 0.05, adjustment_note, transform=ax3.transAxes, fontsize=10,
              va='bottom', bbox=dict(facecolor='white', edgecolor='none', alpha=0.9))
    ax3.set_title('3. One-factor APT pricing\n'
                  f"Alpha: {data['expected_alpha']:.1f}% | "
                  f"Beta: {data['expected_beta']:.1f}%",
                  fontsize=11, fontweight='normal')
    ax3.set_xlabel('Factor beta')
    ax3.set_ylabel('Expected return (%)')
    ax3.set_xlim(-0.1, 2.3)
    ax3.set_xticks(np.arange(0, 2.1, 0.5))

    for ax in (ax1, ax2):
        ax.set_xlabel('Factor surprise (pp)')
        ax.set_ylabel('Realized return (%)')
        ax.set_xlim(-3.2, 3.2)
        ax.set_xticks(np.arange(-3, 4))
    for ax in axes:
        # Fixed across redraws for the current slider ranges and seeded draws.
        # Plotted values range from about -13.89% to 35.05%.
        ax.set_ylim(-20, 40)
        ax.set_yticks(np.arange(-20, 41, 10))
        ax.axhline(0, color='#666666', linewidth=0.7)
        ax.set_axisbelow(True)
        ax.grid(axis='both', linestyle=':', alpha=0.4)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15),
                  ncol=2, fontsize=9, frameon=False)
    return fig


def make_apt_dashboard():
    def slider(value, lower, upper, step, description):
        return widgets.FloatSlider(
            value=value, min=lower, max=upper, step=step,
            description=description, readout_format='.1f', continuous_update=False,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='98%'))

    beta_control = slider(1.0, 0.5, 2.0, 0.1, 'Factor beta:')
    residual_control = slider(5.0, 0.0, 10.0, 0.5, 'Residual SD (pp):')
    gap_control = slider(3.0, -5.0, 5.0, 0.5, 'Beta - Alpha gap (pp):')
    controls = [beta_control, residual_control, gap_control]
    plot_output = widgets.Output()

    def update(change=None):
        data = calculate_apt_scenario(*(control.value for control in controls))
        with plot_output:
            clear_output(wait=True)
            fig = plot_apt_scenario(data)
            plt.show()
            plt.close(fig)

    for control in controls:
        control.observe(update, names='value')
    dashboard = widgets.VBox([
        widgets.HTML('<h3 style="font-weight: normal;">APT: diversification, arbitrage, and pricing</h3>'),
        widgets.HTML(
            '<p>Alpha and Beta share the selected factor beta and have zero residual risk.<br>'
            'The residual-risk slider applies only to the single asset. '
            'Arbitrage trades use equal dollar positions and assume frictionless trading.</p>'),
        widgets.HBox([widgets.VBox([control], layout=widgets.Layout(width='33%'))
                      for control in controls]),
        plot_output,
    ])
    update()
    return dashboard


apt_dashboard = make_apt_dashboard()
display(apt_dashboard)


### The General APT Equation

*   Multifactor Generalization
    *   Stephen Ross (1976) generalized the arbitrage logic to an economy with $K$ systematic risk factors.
    *   The APT pricing relation links expected excess returns to sensitivities to the systematic factors.

*   The Pricing Equation (Excess Returns)
    *   For asset $i$, the APT pricing relation is:
    $$
    E[R_i] = \beta_{i1}\lambda_1 + \beta_{i2}\lambda_2 + \dots + \beta_{iK}\lambda_K
    $$

*   Definitions
    *   $\beta_{ik}$: The sensitivity of asset $i$ to factor $k$ (factor loading).
    *   $\lambda_k$: The Factor Risk Premium associated with factor $k$.
        *   It represents the extra return required for holding a portfolio with a beta of 1 on factor $k$ and 0 on all others.

*   The total expected excess return is the sum of the risk premiums for each systematic risk factor the asset is exposed to.

---

### Factor-Mimicking Portfolios (Tracking Portfolios)

*   Estimating Factor Risk Premiums
    *   The APT establishes the relationship between expected returns and factor sensitivities, but it does not quantify the risk premiums ($\lambda_k$) directly.
    *   Since theoretical factors (e.g., inflation) are often non-traded, we estimate premiums using factor-mimicking portfolios.

*   Construction Properties (The Ideal)
    *   Normalization: $\beta_k=1$ for the target factor.
    *   Zero exposure to other factors: $\beta_j=0$ for $j \neq k$.
    *   Diversification: idiosyncratic risk is negligible.

*   The Pricing Result
    *   Under these conditions, the portfolio's expected excess return equals $\lambda_k$.
    $$
    E[R_{P,k}] = \lambda_k
    $$

---

### CAPM vs. APT

| Aspect | CAPM | APT |
| :--- | :--- | :--- |
| Pricing mechanism | Equilibrium from mean-variance portfolio choice | Absence of arbitrage in a factor model |
| Investor assumptions | Mean-variance preferences and common expectations | No particular utility function or normal return distribution required |
| Market portfolio | Mean-variance efficient; investors hold it as their risky portfolio | No special role for the market portfolio |
| Role of diversification | Prices systematic market risk | Requires enough securities to diversify idiosyncratic risk |
| Applicability | Every asset satisfies the pricing equation under the model's assumptions | Exact in the ideal diversified limit; individual-asset pricing is approximate |

An individual stock can retain idiosyncratic risk, so a deviation from the APT pricing relation need not create a risk-free arbitrage opportunity.

---


### The Fama-French 3-Factor Model (FF3)

APT does not identify the priced factors. Empirical models propose observable factors based on patterns in returns.

*   Empirical Motivation
    *   Fama and French (1993) added size and value factors to capture historical patterns in average returns that market beta did not fully explain.

*   The Regression Equation (Attribution)
    *   To measure risk exposures, we estimate the following time-series regression using excess returns ($R_{i,t} \equiv r_{i,t} - r_{f,t}$):
    $$
    R_{i,t} = \alpha_i + \beta_{i,M} R_{M,t} + \beta_{i,SMB} SMB_t + \beta_{i,HML} HML_t + e_{i,t}
    $$

*   The Pricing Equation (Equilibrium)
    *   If FF3 prices the asset correctly, the intercept $\alpha_i=0$, so its expected excess return is:
    $$
    E[R_i] = \beta_{i,M} \lambda_M + \beta_{i,SMB} \lambda_{SMB} + \beta_{i,HML} \lambda_{HML}
    $$
    *   Factor premiums are expected factor returns: $\lambda_M=E[R_M]$, $\lambda_{SMB}=E[SMB]$, and $\lambda_{HML}=E[HML]$.

---

### Fama-French Factors: Meaning and Construction

| Factor | Construction | Interpretation of positive exposure |
| :--- | :--- | :--- |
| Market ($R_M=r_M-r_f$) | Broad market return minus the risk-free rate | Positive sensitivity to market excess returns |
| SMB (Small Minus Big) | Long small firms; short large firms | A tilt toward small-cap stocks |
| HML (High Minus Low) | Long high book-to-market firms; short low book-to-market firms | A tilt toward value stocks |

* In FF3, stocks are grouped by size and book-to-market into six portfolios (a $2\times3$ sort).
* SMB averages the small-minus-big spreads across value groups; HML averages the value-minus-growth spreads across size groups.
* These sorts help separate size and value effects, but do not guarantee zero exposure to other factors.

---


### Explaining the Size and Value Premiums

What might explain the historical size and value premiums?

*   Risk-Based View (Rational Pricing)
    *   Size (SMB): Small firms may face systematic financing and liquidity risks that market beta does not fully capture.
    *   Value (HML): Distress and vulnerability to economic downturns may explain why value stocks earn higher average returns.

*   Behavioral View (Mispricing)
    *   Value vs. Growth: Investors may extrapolate past growth too far, overpricing "glamour" stocks and underpricing value stocks.
    *   Correction: As expectations are corrected, value stocks may outperform growth stocks.

---


### The Carhart 4-Factor Model

*   The Momentum Anomaly
    *   Jegadeesh and Titman (1993) documented that past winners tend to outperform past losers over horizons of 3 to 12 months.
    *   FF3 did not explain this momentum effect.

*   Augmenting the Model
    *   Carhart (1997) added momentum to FF3.
    *   WML (Winners Minus Losers): the return on a portfolio long past winners and short past losers.

*   The Regression Equation
    $$
    R_{i,t} = \alpha_i + \beta_{i,M} R_{M,t} + \beta_{i,SMB} SMB_t + \beta_{i,HML} HML_t + \beta_{i,WML} WML_t + e_{i,t}
    $$

*   Application
    *   A widely used benchmark for evaluating mutual fund performance after controlling for common factor exposures, including momentum.

---


### The Fama-French 5-Factor Model

Fama and French (2015) added operating profitability and investment factors to FF3.

FF5 uses market, SMB, HML, RMW, and CMA; it does not include momentum.

| New factor | Construction |
| :--- | :--- |
| RMW (Robust Minus Weak) | Long firms with robust operating profitability; short firms with weak operating profitability |
| CMA (Conservative Minus Aggressive) | Long firms with low total asset growth; short firms with high total asset growth |

In their original US sample, HML's average return was captured by its exposures to the other factors; this redundancy need not hold in other samples.

---


In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import datetime
import warnings

# --- 1. UI Configuration ---
SECTOR_TICKERS = ['XLK', 'XLV', 'XLF', 'XLE', 'XLY', 'XLP', 'XLI', 'XLU', 'XLB', 'XLRE', 'XLC', '^SP500TR']
TICKER_MAP = {
    'XLK': 'Tech (XLK)', 'XLV': 'Health (XLV)', 'XLF': 'Financials (XLF)',
    'XLE': 'Energy (XLE)', 'XLY': 'Discretionary (XLY)', 'XLP': 'Staples (XLP)',
    'XLI': 'Industrial (XLI)', 'XLU': 'Utilities (XLU)', 'XLB': 'Materials (XLB)',
    'XLRE': 'Real Estate (XLRE)', 'XLC': 'Comm (XLC)', '^SP500TR': 'S&P 500 TR'
}

# Widgets
style = {'description_width': 'initial'}
start_date_widget = widgets.DatePicker(description='Start Date:', value=datetime.date(2010, 1, 1))
end_date_widget = widgets.DatePicker(description='End Date:', value=datetime.date(2025, 12, 31))
asset_checkboxes = {t: widgets.Checkbox(value=(t in ['XLK', 'XLE', 'XLF']), description=TICKER_MAP.get(t, t), indent=False) for t in SECTOR_TICKERS}
FACTOR_LABELS = {'MKT': 'Market (MKT)', 'SMB': 'Size (SMB)', 'HML': 'Value (HML)', 'MOM': 'Momentum (MOM)'}
factor_checkboxes = {f: widgets.Checkbox(value=True, description=l, indent=False) for f, l in FACTOR_LABELS.items()}
run_button = widgets.Button(description="Run Factor Analysis", button_style='success', icon='chart-line')
output_area = widgets.Output()

# --- 2. Core Analysis Logic ---

def fetch_and_analyze(b=None):
    with output_area:
        clear_output(wait=True)
        selected_tickers = [t for t, cb in asset_checkboxes.items() if cb.value]
        selected_factors = [f for f, cb in factor_checkboxes.items() if cb.value]
        start, end = start_date_widget.value, end_date_widget.value

        if not selected_tickers:
            display(HTML("<p style='color:red;'>Select at least one asset.</p>"))
            return
        if not selected_factors:
            display(HTML("<p style='color:red;'>Select at least one factor.</p>"))
            return
        if start is None or end is None or start >= end:
            display(HTML("<p style='color:red;'>Start date must be before end date.</p>"))
            return

        # Include only whole calendar months within the requested dates.
        first_month = pd.Period(start, freq='M')
        last_month = pd.Period(end, freq='M')
        if start.day != 1:
            first_month += 1
        if end.day != last_month.days_in_month:
            last_month -= 1
        last_month = min(last_month, pd.Period(datetime.date.today(), freq='M') - 1)
        if first_month > last_month:
            display(HTML("<p style='color:red;'>Select a period containing at least one completed calendar month.</p>"))
            return

        # Fetch the preceding month's close to calculate the first return.
        price_start = (first_month - 1).start_time.date()
        price_end = (last_month + 1).start_time.date()
        factor_start = first_month.start_time.date()
        factor_end = last_month.end_time.date()

        print('1. Downloading Asset Data (Yahoo)...')
        try:
            # Use the ticker-grouped adjusted-close workflow from Lecture 6.
            # Yahoo excludes the end date; auto_adjust=True makes Close adjusted.
            downloaded = yf.download(
                selected_tickers, start=price_start, end=price_end,
                interval='1mo', group_by='ticker', auto_adjust=True,
                actions=False, progress=False
            )
            if downloaded is None or downloaded.empty:
                raise ValueError('Yahoo returned no monthly adjusted-close data.')

            series = {}
            for ticker in selected_tickers:
                if isinstance(downloaded.columns, pd.MultiIndex):
                    if ticker not in downloaded.columns.get_level_values(0):
                        continue
                    item = downloaded[ticker]
                elif len(selected_tickers) == 1:
                    item = downloaded
                else:
                    raise ValueError('Yahoo returned an unexpected column layout.')
                if 'Close' in item.columns and item['Close'].notna().any():
                    series[ticker] = item['Close']
            missing = [ticker for ticker in selected_tickers if ticker not in series]
            if missing:
                raise ValueError('Missing adjusted closes: ' + ', '.join(missing))

            prices = pd.DataFrame(series).sort_index()
            prices.index = pd.DatetimeIndex(prices.index).tz_localize(None).to_period('M')
            prices = prices[~prices.index.duplicated(keep='last')]
            # A missing close invalidates both adjoining monthly returns.
            prices = prices.reindex(pd.period_range(first_month - 1, last_month, freq='M'))
            asset_returns = prices.pct_change(fill_method=None).replace(
                [np.inf, -np.inf], np.nan).loc[first_month:last_month]
        except Exception as e:
            display(HTML(f"<p style='color:red;'>Yahoo Error: {e}</p>"))
            return

        print('2. Downloading Factor Data (Kenneth French)...')
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', FutureWarning)
                ff3 = web.DataReader(
                    'F-F_Research_Data_Factors', 'famafrench',
                    start=factor_start, end=factor_end)[0]
                factor_frames = [ff3]
                if 'MOM' in selected_factors:
                    mom = web.DataReader(
                        'F-F_Momentum_Factor', 'famafrench',
                        start=factor_start, end=factor_end)[0]
                    factor_frames.append(mom)
            factors = pd.concat(factor_frames, axis=1) / 100.0
            factors.columns = factors.columns.str.strip()
            factors.rename(columns={'Mkt-RF': 'MKT', 'Mom': 'MOM'}, inplace=True)
            # Unselected factors must not shorten the estimation sample.
            factors = factors[selected_factors + ['RF']]
        except Exception as e:
            display(HTML(f"<p style='color:red;'>Fama-French Error: {e}</p>"))
            return

        # Use identical months for every asset so the estimates are comparable.
        aligned_data = (asset_returns.join(factors, how='inner')
                        .replace([np.inf, -np.inf], np.nan).dropna().sort_index())
        nobs = len(aligned_data)
        if nobs == 0:
            display(HTML("<p style='color:red;'>No complete monthly observations shared by the selected assets and factors.</p>"))
            return
        X = sm.add_constant(aligned_data[selected_factors], has_constant='add')
        if nobs <= X.shape[1]:
            display(HTML(
                f"<p style='color:red;'>Only {nobs} common monthly observations. "
                f"At least {X.shape[1] + 1} are needed for {len(selected_factors)} "
                "factors plus alpha. Extend the period or select fewer factors.</p>"))
            return
        if np.linalg.matrix_rank(X.to_numpy()) < X.shape[1]:
            display(HTML(
                "<p style='color:red;'>Selected factors are constant or linearly dependent "
                "in this sample. Extend the period or select fewer factors.</p>"))
            return

        alpha_label = 'Monthly alpha (\u03b1)'
        results_list = []
        for ticker in selected_tickers:
            y = aligned_data[ticker] - aligned_data['RF']
            model = sm.OLS(y, X).fit()
            res = {'Asset': TICKER_MAP.get(ticker, ticker),
                   alpha_label: model.params['const'],
                   'R-Squared': model.rsquared, 'Adj R-Squared': model.rsquared_adj}
            for f in selected_factors:
                res[f'Beta {f}'] = model.params[f]
            results_list.append(res)

        clear_output(wait=True)
        results_df = pd.DataFrame(results_list).set_index('Asset')
        beta_columns = [f'Beta {f}' for f in selected_factors]
        format_dict = {alpha_label: '{:.4%}', 'R-Squared': '{:.4f}', 'Adj R-Squared': '{:.4f}'}
        format_dict.update({col: '{:.2f}' for col in beta_columns})

        # Symmetric color limits put zero at the neutral midpoint.
        color_limit = max(float(results_df[beta_columns].abs().to_numpy().max()), 1e-12)
        styler = results_df.style.format(format_dict).background_gradient(
            cmap='coolwarm', subset=beta_columns, axis=None,
            vmin=-color_limit, vmax=color_limit)
        sample_start = aligned_data.index.min().strftime('%b %Y')
        sample_end = aligned_data.index.max().strftime('%b %Y')
        display(HTML('<h3 style="font-weight: normal;">Factor Analysis</h3>'))
        display(HTML(
            f'<p>Common sample: {sample_start} &ndash; {sample_end}; '
            f'{nobs} monthly observations.<br>All assets use the same complete months.</p>'))
        display(styler)
        display(HTML('<p>Table shading: blue = negative beta; red = positive beta. '
                     'Bar colors identify factors.</p>'))

        factor_colors = {'MKT': '#0072B2', 'SMB': '#D55E00',
                         'HML': '#009E73', 'MOM': '#CC79A7'}
        fig, ax = plt.subplots(figsize=(max(12, len(selected_tickers)), 6))
        results_df[beta_columns].plot(
            kind='bar', ax=ax, width=0.8,
            color=[factor_colors[f] for f in selected_factors])
        ax.set_title('Factor Exposures (Betas)', fontsize=14, fontweight='normal')
        ax.set_ylabel('Factor beta')
        ax.set_xlabel('')
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_axisbelow(True)
        ax.grid(axis='y', linestyle=':', alpha=0.5)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.0),
                  ncol=len(selected_factors), frameon=False)
        plt.xticks(rotation=30 if len(selected_tickers) > 4 else 0,
                   ha='right' if len(selected_tickers) > 4 else 'center')
        ax.margins(y=0.15)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        display(HTML("""
        <div style="background-color: #f9f9f9; padding: 10px; border-left: 5px solid #2196F3; margin-top: 10px;">
        Interpretation Guide:
        <ul>
            <li>Monthly alpha: Average excess return unexplained by the selected factors. A positive estimate alone does not establish skill; model choice and estimation uncertainty matter.</li>
            <li>Market beta: Above 1 = amplified market exposure; between 0 and 1 = lower exposure; below 0 = inverse exposure.</li>
            <li>SMB (Size): Positive = small-cap tilt; negative = large-cap tilt.</li>
            <li>HML (Value): Positive = value tilt; negative = growth tilt.</li>
            <li>MOM (Momentum): Positive = exposure to past winners over losers; negative = the reverse.</li>
        </ul>
        </div>
        """))

# --- 3. Assemble UI ---
run_button.on_click(fetch_and_analyze)
asset_grid = widgets.GridBox(list(asset_checkboxes.values()), layout=widgets.Layout(grid_template_columns="repeat(4, 150px)"))
factor_grid = widgets.HBox(list(factor_checkboxes.values()))
ui = widgets.VBox([widgets.HTML("<h3>Factor Model Analyzer</h3>"), widgets.HBox([start_date_widget, end_date_widget]), widgets.Label("Only completed calendar months within these dates are used."), widgets.Label("Select Factors:"), factor_grid, widgets.Label("Select Assets:"), asset_grid, run_button, output_area])
display(ui)

### Smart Beta and Factor Investing

Some apparent alpha can be explained by exposure to common return factors, such as size, value, and momentum.

Smart beta funds use transparent rules to select and/or weight securities by characteristics such as valuation ratios, profitability, or volatility.

| Implementation | Portfolio construction |
| :--- | :--- |
| Academic factors such as SMB and HML | Zero-net-investment long-short portfolios |
| Smart beta funds (often ETFs) | Typically long-only portfolios tilted toward selected characteristics |

For example, a typical value ETF overweights value stocks relative to a market-cap-weighted benchmark without shorting growth stocks.

---


### Multifactor Performance Evaluation

Regress the fund's excess returns on factor returns, such as market excess returns, SMB, HML, and WML.

| Measure | Interpretation |
| :--- | :--- |
| $R^2$ | Share of excess-return variation explained by the model. High $R^2$ does not rule out positive alpha. |
| Factor betas | Exposure to each factor, such as a tilt toward small-cap or value stocks. |
| Alpha ($\alpha$) | Regression intercept: average return unexplained by the included factors. |

A statistically significant positive alpha provides evidence of outperformance relative to the chosen model; it does not establish skill by itself.

Evaluate performance after fees against an appropriate factor benchmark.

---


### Question 1

Alpha and Beta make the following claims about asset pricing:

1. Alpha: "CAPM requires identical risk aversion, so all investors hold the same mix of risky assets and the risk-free asset."

2. Beta: "If a stock's expected return exceeds the APT prediction, hedging its factor exposures guarantees an arbitrage profit."

* Assess each claim and justify your answer. 

* Explain the roles of the market portfolio in CAPM and diversification in APT.

---


### Question 2

Fund Gamma reports the following regression estimates using monthly excess returns after fees over the same sample:

| Model | Monthly alpha | SMB loading | HML loading |
| :--- | ---: | ---: | ---: |
| CAPM | 0.30% | Not included | Not included |
| FF3 | 0.00% | 0.60 | 0.40 |

Gamma's manager claims that the positive CAPM alpha demonstrates stock-picking skill.

1. What investment style do the FF3 loadings suggest?

2. Assess the manager's claim. Does the FF3 result prove that the manager has no skill?

---


### Question 3

In a one-factor economy, Delta and Epsilon are portfolios with zero idiosyncratic risk. Zeta is risk-free. The table reports one-year total returns.

| Portfolio | Factor beta | Expected return |
| :--- | ---: | ---: |
| Delta | 1.5 | 10% |
| Epsilon | 1.0 | 9% |
| Zeta | 0.0 | 3% |

Assume unrestricted short selling, borrowing at the risk-free rate, and no transaction costs.

1. Compare the factor risk premiums implied by Delta and Epsilon. Are their expected returns consistent with no-arbitrage pricing?

2. Construct a zero-net-investment, zero-factor-exposure trade with a \$100 position in Delta. Specify whether each position is long or short and calculate the certain profit after one year.

---


### Question 4

Stock Eta is priced by a two-factor model. The annual risk-free rate is 3%.

| Factor | Annual risk premium | Eta's loading |
| :--- | ---: | ---: |
| GDP growth | 5% | 1.2 |
| Inflation | -1% | 0.5 |

1. Calculate Eta's expected annual total return.

2. GDP growth was forecast at 3% but ended at 1%; inflation was forecast at 2% but ended at 4%. Calculate Eta's realized total return, assuming zero idiosyncratic surprise. Explain why the inflation factor has opposite effects in the two calculations.

Measure macroeconomic surprises as actual minus forecast, in percentage points.

---


### Question 5

Fund Theta's monthly excess returns are fitted by OLS:

$$
R_{\mathrm{Theta},t}=\hat{\alpha}+1.1R_{M,t}-0.5SMB_t+0.4HML_t+\hat{e}_t.
$$

The table reports monthly arithmetic averages from the same sample. Fund returns are after fees.

| Series | Monthly average |
| :--- | ---: |
| Market excess return | 0.60% |
| SMB | 0.20% |
| HML | 0.35% |
| Risk-free return | 0.20% |
| Theta's total return | 0.75% |

1. Calculate Theta's monthly benchmark excess return and total return, setting alpha to zero.

2. Find its monthly alpha and annualize it using $12\times$ monthly alpha.

3. Interpret the SMB and HML loadings. What investment style do they suggest?

---


### Question 6

You own \$100 of Portfolio Iota, financed entirely with your own equity. Fund Kappa is available as a hedge:

| Investment | Market beta | Interest-rate beta |
| :--- | ---: | ---: |
| Iota | 1.1 | 1.8 |
| Kappa | 0.2 | -0.6 |

You retain all of Iota and finance any purchase of Kappa entirely by borrowing at a fixed risk-free rate. Assume the loan has zero exposure to both factors.

1. How much Kappa should you buy to eliminate interest-rate exposure?

2. Calculate the market beta of the financed position relative to your net equity. Does the hedge reduce market exposure?

---


### What is next?

*   The Efficient Market Hypothesis & Behavioral Finance    
    *   The Efficient Market Hypothesis
    *   The Behavioral Critique
    *   Empirical Evidence on Security Returns
    *   Reading(s): BKM Ch. 11, 12.1 &13
    
*   Suggested Problems
    *   Ch.10: 1, 4, 5, 10, and 18.
    *   Ch.10-CFA Problems: 1,2, and 8.

---


### ESG and Equilibrium Asset Pricing

*   Motivation
    *   How does investor preference for Environmental, Social, and Governance (ESG) attributes affect stock prices?
    *   Pástor, Stambaugh, and Taylor (2021) model an economy with "Green" investors (who derive utility from holding green assets) and "Neutral" investors.

*   The Modified CAPM
    *   Expected excess returns reflect market risk and investor preferences for ESG:
    $$
    E[R_i] = \beta_{i,M} \lambda_M - \beta_{i,ESG} \lambda_{Taste}
    $$
    *   $\beta_{i,ESG}=g_i$: Asset $i$'s ESG characteristic relative to the market, equal to its loading on the model's normalized ESG factor.
    *   $\lambda_{Taste}>0$: Expected-return discount per unit of ESG exposure due to investor preferences.
    *   Holding market beta fixed, greener assets have lower expected excess returns.

*   Realized vs. Expected Returns
    *   Green assets can outperform despite lower expected returns.
    *   Unexpected increases in demand for green investments raise prices, boosting realized returns while lowering future expected returns.

---

### Constructing Factor Portfolios: The Theoretical Approach

*   The Objective
    *   To measure a Factor Risk Premium ($\lambda_k$), we need a portfolio that tracks *only* that specific source of risk, hedging out all others.
    *   Definition: A well-diversified portfolio with $\beta_k = 1$ (unit sensitivity to the target factor) and $\beta_j = 0$ (zero sensitivity to all other systematic factors).

*   Step 1: Estimating Sensitivities (Regression)
    *   Run time-series regressions for a large universe of assets ($N$) to estimate their factor exposures:
    $$
    R_{i,t} = \mu_i + \beta_{i,1}F_{1,t} + \beta_{i,2}F_{2,t} + \dots + e_{i,t}
    $$
    *   With mean-zero factor surprises and residuals, the intercept is expected excess return: $\mu_i=E(R_i)$.

*   Step 2: Solving for Weights
    *   Choose risky-asset weights ($w_i$) to meet the exposure targets while minimizing residual risk:
        1.  Target Exposure: $\sum_{i=1}^{N} w_i \beta_{i,k} = 1$
        2.  Orthogonality: $\sum_{i=1}^{N} w_i \beta_{i,j} = 0$ for all $j \neq k$
        3.  Diversification: Minimize idiosyncratic variance $\sum_{i=1}^{N} w_i^2 \sigma^2(e_i)$
    *   Set the risk-free weight to $w_f=1-\sum_{i=1}^{N}w_i$; $w_f<0$ means borrowing.

---